# AndinaLog 03B | Inventory_Tracking | Tratamiento v2

Contrato didáctico: datos originales visibles, decisiones trazables y tres salidas CSV.


In [ ]:
from pathlib import Path
import hashlib
import sys
import pandas as pd
import numpy as np

ENTORNO="auto"  # auto, local, drive
RUTA_PROYECTO_DRIVE="/content/drive/MyDrive/GIAD"
VERSION_DIAGNOSTICO_REQUERIDA="GIAD-M3-S4-INVENTORY-diagnostico-didactico-v2"
VERSION_TRATAMIENTO="GIAD-M3-S4-INVENTORY-tratamiento-didactico-v2"
MIN_LOTES_VIDA_UTIL=20
COLUMNAS_BRONZE=["movimiento_id","lote_id","producto_id","centro_distribucion",
    "fecha_ingreso","fecha_salida","fecha_vencimiento","cantidad_ingreso",
    "cantidad_salida","cantidad_merma","dias_en_almacen","costo_unitario_bob"]
CENTROS={"Cochabamba","La Paz","Santa Cruz","Oruro","Tarija"}

def encontrar_raiz():
    if ENTORNO=="drive" or (ENTORNO=="auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz=Path(RUTA_PROYECTO_DRIVE)
        if not (raiz/"proyecto-integrador/01_diagnostico/andinalog_inventory_tracking/salidas/andinalog_inventory_tracking_didactico_v2_diagnosticado.csv").is_file():
            raise FileNotFoundError(raiz)
        return raiz
    for carpeta in [Path.cwd(),*Path.cwd().parents]:
        if (carpeta/"proyecto-integrador/01_diagnostico/andinalog_inventory_tracking/salidas/andinalog_inventory_tracking_didactico_v2_diagnosticado.csv").is_file():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz de practicasNotebookColab")

RAIZ=encontrar_raiz()
RUTA_BRONZE=RAIZ/"datasets/AndinaLog_03B_Bronce/andinalog_inventory_tracking.csv"
ENTRADA=RAIZ/"proyecto-integrador/01_diagnostico/andinalog_inventory_tracking/salidas/andinalog_inventory_tracking_didactico_v2_diagnosticado.csv"
SALIDAS=RAIZ/"proyecto-integrador/02_tratamiento/andinalog_inventory_tracking/salidas"
df=pd.read_csv(ENTRADA,dtype="string",encoding="utf-8-sig",keep_default_na=False)
bronze=pd.read_csv(RUTA_BRONZE,dtype="string",encoding="utf-8-sig",keep_default_na=False)
requeridas=["fila_bronze","en_cuarentena",*COLUMNAS_BRONZE]
requeridas += [f"{c}_{s}" for c in COLUMNAS_BRONZE for s in ("en_cuarentena","motivo")]
faltan=sorted(set(requeridas)-set(df.columns))
if faltan: raise ValueError(f"Faltan columnas del diagnóstico: {faltan}")
if list(bronze.columns)!=COLUMNAS_BRONZE: raise ValueError("Esquema Bronze inesperado")
if len(df)!=len(bronze) or df["fila_bronze"].duplicated().any():
    raise ValueError("Filas diagnosticadas incompletas o duplicadas")
if not df["fila_bronze"].eq(pd.Series(range(1,len(df)+1),dtype="string")).all():
    raise ValueError("Orden Bronze inesperado")
pd.testing.assert_frame_equal(df[COLUMNAS_BRONZE],bronze)
huella=hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest()
original=df.copy(deep=True)
df=df.rename(columns={"en_cuarentena":"en_cuarentena_diagnostico"})
print("Entrada:",len(df),"| Cuarentena diagnóstica:",int(df["en_cuarentena_diagnostico"].eq("True").sum()))


## 1. Preparación determinista

Los campos Bronze se conservan. Las fechas se interpretan como fechas de calendario, sin zona horaria. El ID de producto se normaliza solo como escritura; su ausencia en Productos Silver de cobertura parcial no invalida el lote.


In [ ]:
df["acciones_tratamiento"]=""
df["motivos_tratamiento"]=""
def anotar(mascara,accion,motivo):
    mascara=pd.Series(mascara,index=df.index).fillna(False).astype(bool)
    for c,texto in [("acciones_tratamiento",accion),("motivos_tratamiento",motivo)]:
        previo=df.loc[mascara,c]
        df.loc[mascara,c]=previo.where(previo.eq(""),previo+" | ")+texto

def fecha_tratada(columna):
    valor=df[columna].str.strip()
    iso=valor.str.fullmatch(r"\d{4}-\d{2}-\d{2}").fillna(False)
    alt=valor.str.fullmatch(r"\d{2}/\d{2}/\d{4}").fillna(False)
    f=pd.to_datetime(valor.where(iso),format="%Y-%m-%d",errors="coerce").fillna(
      pd.to_datetime(valor.where(alt),format="%d/%m/%Y",errors="coerce"))
    df[f"{columna}_tratada"]=f.dt.strftime("%Y-%m-%d").fillna("")
    normalizada=alt & f.notna()
    anotar(normalizada,f"NORMALIZAR_{columna.upper()}","Fecha válida DD/MM/AAAA convertida a AAAA-MM-DD")
    return f,normalizada

ingreso,ing_norm=fecha_tratada("fecha_ingreso")
salida,sal_norm=fecha_tratada("fecha_salida")
vencimiento,ven_norm=fecha_tratada("fecha_vencimiento")
df["producto_id_tratado"]=df["producto_id"].str.strip().str.upper()
producto_normalizado=df["producto_id_tratado"].ne(df["producto_id"])
anotar(producto_normalizado,"NORMALIZAR_PRODUCTO_ID","Espacios y mayúsculas normalizados")

for c in ["cantidad_ingreso","cantidad_salida","cantidad_merma","dias_en_almacen","costo_unitario_bob"]:
    df[f"{c}_tratada"]=pd.to_numeric(df[c].str.strip(),errors="coerce")

# Una equivalencia se determina sobre las 12 columnas, después de preparar fecha e ID.
columnas_preparadas=["movimiento_id","lote_id","producto_id_tratado","centro_distribucion",
    "fecha_ingreso_tratada","fecha_salida_tratada","fecha_vencimiento_tratada",
    "cantidad_ingreso_tratada","cantidad_salida_tratada","cantidad_merma_tratada",
    "dias_en_almacen_tratada","costo_unitario_bob_tratada"]
firma=pd.util.hash_pandas_object(df[columnas_preparadas],index=False)
copia=df.duplicated(columnas_preparadas,keep="first")
conflicto=pd.Series(False,index=df.index)
for clave in ["movimiento_id","lote_id"]:
    valor=df[clave].str.strip()
    variantes=firma.groupby(valor,dropna=False).transform("nunique")
    conflicto |= valor.ne("") & valor.duplicated(keep=False) & variantes.gt(1)
copia &= ~conflicto
anotar(copia,"EXCLUIR_COPIA","Copia posterior idéntica después de preparar fechas e ID")
anotar(conflicto,"CUARENTENA_CONFLICTO","Misma clave con atributos contradictorios")


## 2. Imputación de vencimiento por producto

La vida útil se calcula solo con lotes observados, válidos, sin duplicidad/conflicto, fechas de ingreso y vencimiento coherentes y `producto_id_tratado` válido. Debe haber al menos 20 lotes y una sola duración en días para ese producto. La imputación se marca por fila y el valor Bronze permanece vacío. Un vencimiento estimado habilita Silver **analítico**, pero no un KPI que exija vencimiento observado ni una decisión operativa sin verificación del lote.


In [ ]:
vida_dias=(vencimiento-ingreso).dt.days
producto_valido=df["producto_id_tratado"].str.fullmatch(r"PROD-\d{3}").fillna(False)
base_vida=(~copia & ~conflicto & producto_valido & ingreso.notna() &
           vencimiento.notna() & vida_dias.gt(0))
historico=pd.DataFrame({"producto_id":df.loc[base_vida,"producto_id_tratado"],
                        "vida_dias":vida_dias.loc[base_vida]})
resumen_vida=historico.groupby("producto_id")["vida_dias"].agg(["count","nunique","first"])
aprobados=resumen_vida.loc[(resumen_vida["count"]>=MIN_LOTES_VIDA_UTIL) &
                           resumen_vida["nunique"].eq(1),"first"]
df["vida_util_producto_dias_usada"]=pd.to_numeric(df["producto_id_tratado"].map(aprobados),errors="coerce")
falta_vencimiento=df["fecha_vencimiento"].str.strip().eq("")
df["vencimiento_imputado"]=(falta_vencimiento & ingreso.notna() & producto_valido &
    df["vida_util_producto_dias_usada"].notna() & ~copia & ~conflicto)
estimado=ingreso+pd.to_timedelta(df["vida_util_producto_dias_usada"],unit="D")
df.loc[df["vencimiento_imputado"],"fecha_vencimiento_tratada"]=estimado.loc[
    df["vencimiento_imputado"]].dt.strftime("%Y-%m-%d")
anotar(df["vencimiento_imputado"],"IMPUTAR_VENCIMIENTO",
       "Fecha estimada con vida útil única del producto y al menos 20 lotes observados")
vencimiento_final=pd.to_datetime(df["fecha_vencimiento_tratada"],format="%Y-%m-%d",errors="coerce")


## 3. Decisión final y marcas analíticas

`riesgo_salida_post_vencimiento` detecta hechos de negocio y no causa cuarentena por sí solo. `apto_kpi_vencimiento_observado` exige una fecha de vencimiento realmente registrada. No se imputa una cantidad textual, una merma negativa ni una fecha imposible.


In [ ]:
df["motivo_cuarentena_final"]=""
def cuarentena_si(mascara,motivo):
    mascara=pd.Series(mascara,index=df.index).fillna(False).astype(bool)
    previo=df.loc[mascara,"motivo_cuarentena_final"]
    df.loc[mascara,"motivo_cuarentena_final"]=previo.where(previo.eq(""),previo+" | ")+motivo

cuarentena_si(copia,"Copia excluida del Silver")
cuarentena_si(conflicto,"Clave de movimiento o lote en conflicto")
cuarentena_si(~df["movimiento_id"].str.fullmatch(r"MOV-\d{6}").fillna(False),"ID de movimiento inválido")
cuarentena_si(~df["lote_id"].str.fullmatch(r"LOT-\d{4}-\d{5}").fillna(False),"ID de lote inválido")
cuarentena_si(~producto_valido,"ID de producto inválido")
cuarentena_si(~df["centro_distribucion"].isin(CENTROS),"Centro inválido")
cuarentena_si(ingreso.isna(),"Fecha de ingreso inválida")
cuarentena_si(df["fecha_salida"].str.strip().ne("") & salida.isna(),"Fecha de salida inválida")
cuarentena_si(vencimiento_final.isna(),"Vencimiento sin valor observado ni estimación confiable")
cuarentena_si(salida.notna() & ingreso.notna() & salida.lt(ingreso),"Salida anterior al ingreso")
cuarentena_si(vencimiento_final.notna() & ingreso.notna() & vencimiento_final.lt(ingreso),
             "Vencimiento anterior al ingreso")

for c in ["cantidad_ingreso","cantidad_salida","cantidad_merma","dias_en_almacen"]:
    n=df[f"{c}_tratada"]
    cuarentena_si(n.isna() | n.lt(0) | n.mod(1).ne(0),f"{c} inválida")
cuarentena_si(df["cantidad_ingreso_tratada"].eq(0),"Ingreso debe ser mayor que cero")
cuarentena_si(df["costo_unitario_bob_tratada"].isna() |
             df["costo_unitario_bob_tratada"].le(0),"Costo unitario inválido")

saldo=(df["cantidad_ingreso_tratada"]-df["cantidad_salida_tratada"]-
       df["cantidad_merma_tratada"])
df["saldo_unidades_inventario"]=saldo.where(saldo.ge(0))
cuarentena_si(saldo.lt(0) & df["cantidad_ingreso_tratada"].ge(0) &
             df["cantidad_salida_tratada"].ge(0) & df["cantidad_merma_tratada"].ge(0),
             "Salida más merma supera ingreso")
df["riesgo_salida_post_vencimiento"]=(salida.notna() & vencimiento_final.notna() &
                                      salida.gt(vencimiento_final))
df["riesgo_basado_en_vencimiento_imputado"]=df["riesgo_salida_post_vencimiento"] & df["vencimiento_imputado"]
df["en_cuarentena_final"]=df["motivo_cuarentena_final"].ne("")
df["decision_tratamiento"]=np.where(df["en_cuarentena_final"],"CUARENTENA","SILVER")
df["apto_kpi_vencimiento_observado"]=(~df["en_cuarentena_final"] &
    ~df["vencimiento_imputado"] & vencimiento.notna())
silver=df.loc[~df["en_cuarentena_final"]].copy()
cuarentena_final=df.loc[df["en_cuarentena_final"]].copy()


## 4. Comprobación y exportación

Silver y cuarentena final forman una partición exacta del diagnóstico. Cada CSV conserva original, diagnóstico y trazabilidad del tratamiento. Las cifras de salida se calculan al ejecutar, no se fijan manualmente.


In [ ]:
pd.testing.assert_frame_equal(df[[c for c in original.columns if c!="en_cuarentena"]],original[[c for c in original.columns if c!="en_cuarentena"]])
assert len(df)==len(silver)+len(cuarentena_final)
assert silver["movimiento_id"].is_unique and silver["lote_id"].is_unique
assert silver["motivo_cuarentena_final"].eq("").all()
assert cuarentena_final["motivo_cuarentena_final"].ne("").all()
assert silver["fecha_vencimiento_tratada"].ne("").all()
assert not (silver["apto_kpi_vencimiento_observado"] & silver["vencimiento_imputado"]).any()
assert not silver["saldo_unidades_inventario"].lt(0).any()
assert hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest()==huella

SALIDAS.mkdir(parents=True,exist_ok=True)
ruta_silver=SALIDAS/"andinalog_inventory_tracking_didactico_v2_silver.csv"
ruta_cuarentena=SALIDAS/"andinalog_inventory_tracking_didactico_v2_cuarentena_final.csv"
silver.to_csv(ruta_silver,index=False,encoding="utf-8-sig")
cuarentena_final.to_csv(ruta_cuarentena,index=False,encoding="utf-8-sig")
print("Entrada:",len(df),"| Silver:",len(silver),"| Cuarentena final:",len(cuarentena_final))
print("Fechas normalizadas:",int((ing_norm|sal_norm|ven_norm).sum()),
      "| Productos normalizados:",int(producto_normalizado.sum()),
      "| Vencimientos imputados:",int(df["vencimiento_imputado"].sum()),
      "| Copias excluidas:",int(copia.sum()))
print("Riesgos salida posterior a vencimiento:",int(silver["riesgo_salida_post_vencimiento"].sum()))
print("Silver:",ruta_silver)
print("Cuarentena:",ruta_cuarentena)
display(df[["fila_bronze","movimiento_id","fecha_vencimiento","fecha_vencimiento_tratada",
            "vencimiento_imputado","decision_tratamiento","motivo_cuarentena_final"]].tail(10))

# Third treatment output and published distinction between prior and final quarantine.
assert "en_cuarentena_diagnostico" in df and "en_cuarentena" not in df
assert len(silver)+len(cuarentena_final)==len(original)
metricas={"filas_entrada":len(df),"filas_silver":len(silver),
          "filas_cuarentena_final":len(cuarentena_final),
          "filas_recuperadas":int((df["en_cuarentena_diagnostico"].eq("True") & ~df["en_cuarentena_final"]).sum())}
for campo in ["categoria_imputada","temperatura_imputada","vencimiento_imputado", "capacidad_revisar_ficha"]:
    if campo in df: metricas[campo]=int(df[campo].fillna(False).astype(bool).sum())
reporte_calidad=pd.DataFrame([{"metrica":k,"valor":v} for k,v in metricas.items()])
reporte_calidad.to_csv(SALIDAS/("andinalog_inventory_tracking_didactico_v2_reporte_calidad.csv"),index=False,encoding="utf-8-sig")
print(metricas)
